## WCUS Trips cleaning steps
Get the data [here](https://ndclibrary.sec.usace.army.mil/resource?title=2023%20WCUS%20Trips%20-%20All%20Regions%20&documentId=07ddf14b-1522-4c6f-d894-40cd74d58a7f).

1. Filter to get only In/Out/Thru=="Outbound Shipping" or "Inbound Shipping"
2. Filter to get only TrafficCode==1 (aka TrafficName=="Domestic (Trips & Drafts)")
3. Keep only the columns:
    * 'RegionName', 'Up/Down', 'VesselType', 'VesselTypeName', 'VesselDraftFt', 'Trips', 'CompletedYear'
4. Create a few separate data sets: 
    * Filter to get only WaterwayCode==3924 (aka WaterwayName=="Duluth-Superior, MN and WI")
    * Filter to get only RegionName=="GREAT LAKES"
5. Save as something descriptive like "WCUS_Trips_DuluthSuperior_Outbound_Domestic_2014-2023.csv"

## List of Top Iron Ore Outports and Inports

In [1]:
import pandas as pd

In [2]:
# No Thunder Bay. That is in ON.
# Taconite and Silver Bay (last two) are not USACE ports

# Careful! Cities and their harbors are listed separately?
# E.g. 3841 for Marquette Harbor (admin, maintenance) vs. 3844 for Marquette Township ()
# 3619 is the Presque Isle farther east, by Alpena
# 3845 Presque Isle, MI is subset of Marquette

outports = pd.DataFrame({
    'WaterwayCode':[3924, 3926, 3841]#, 3845, 3929, 3928]
    })



In [3]:
# Note also:
# Cleveland-Cliffs Dearborn Works in Michigan
# Great Lakes Steel in Michigan
# These facilities ultimately receive taconite by rail, though

# Make sure to distinguish between receipts and throughports
    # E.g. Chicago is just a rail hub, other places go straight to mills nearby
# Calumet and Chicago included at end just because Calumet is high-tonnage and Chicago is famous
# Not much iron ore though
inports = pd.DataFrame({
    'WaterwayCode': [3738, 3736, 3739, 3204, 3217, 3315, 3220, 3219, 3741, 3747]
    })

## WCUS Trips Cleaning

In [4]:
path = "../data/raw/Trips_AllRegions_10yr_2014-2023.xlsx"

wcus = pd.read_excel(path, sheet_name="Trips_AllRegions_10yr_2014-2023")

In [5]:
# pd.set_option('display.max_rows', None)
# wcus.loc[:, ['WaterwayCode', 'WaterwayName']].drop_duplicates()

In [22]:
wcus.merge(outports, how='right').loc[:, ['WaterwayName', 'WaterwayCode']].drop_duplicates()
# wcus.loc[wcus['WaterwayCode'].isin(outports['WaterwayCode']), 'WaterwayName'].unique()

,WaterwayName,WaterwayCode
0,"Duluth-Superior, MN and WI",3924
1152,"Two Harbors, MN",3926
1752,"Marquette, MI",3841


In [7]:
# wcus.merge(inports, how='right')['WaterwayName'].unique()
wcus.loc[wcus['WaterwayCode'].isin(inports['WaterwayCode']), 'WaterwayName'].unique()

array(['Ashtabula Port Authority, OH', 'Burns Waterway Harbor, IN',
       'Calumet Harbor and River, IL and IN', 'Chicago Harbor, IL',
       'Cleveland-Cuyahoga Port, OH', 'Conneaut Harbor, OH', 'Gary, IN',
       'Indiana Harbor, IN', 'Rouge River, MI',
       'Toledo-Lucas County Port, OH'], dtype=object)

In [26]:
# TODO
# Now filter into chart-ready CSVs
    # At least 12ft draft when in ballast for ore-carrying vessels according to 2020 LCA Annual Report, Fleet Profile section
    # Up/Down is apparently nonsense, disregard
    # Just Duluth-Superior
        # One csv for Dry Cargo Barge
        # One csv for Self-Propelled dry
# Add up trips if multiple ports belong to a USACE Port Statistical Area
min_draft_ft = 25

wcus_all_out = wcus.loc[
        wcus['WaterwayCode'].isin(outports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] == 'Outbound Shipping')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['TrafficCode'] == 1) # Domestic (Trips & Drafts)
            ,
        :
    ]
wcus_all_out#.head(3)

,RegionName,WaterwayCode,WaterwayName,TrafficCode,TrafficName,In/Out/Thru,Up/Down,VesselType,VesselTypeName,VesselDraftFt,Trips,CompletedYear
149578,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,25,30,2014
149579,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,25,14,2015
149580,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,25,17,2016
149581,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,25,11,2017
149582,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,25,9,2018
...,...,...,...,...,...,...,...,...,...,...,...,...
203053,GREAT LAKES,3926,"Two Harbors, MN",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,31,1,2019
203054,GREAT LAKES,3926,"Two Harbors, MN",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,31,1,2020
203055,GREAT LAKES,3926,"Two Harbors, MN",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,31,8,2021
203056,GREAT LAKES,3926,"Two Harbors, MN",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,31,13,2022


In [9]:
cols_keep = ['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'VesselDraftFt', 'Trips', 'CompletedYear']



### Duluth-Superior Outbound Trip Counts (17ft+ draft depth)

In [10]:
trips_duluth = wcus_all_out.loc[
    wcus_all_out['WaterwayName'] == 'Duluth-Superior, MN and WI',
    cols_keep
]

In [11]:
trip_counts_duluth = trips_duluth.groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'CompletedYear']).sum().reset_index()


In [12]:
# trip_counts_duluth.drop(columns=['VesselDraftFt']).to_csv('../data/clean/trip_counts/outbound/duluth.csv')

### All outport trips

In [13]:
wcus.merge(outports, how='right').loc[:, ['WaterwayName', 'WaterwayCode']].drop_duplicates()


,WaterwayName,WaterwayCode
0,"Duluth-Superior, MN and WI",3924
1152,"Two Harbors, MN",3926
1752,"Marquette, MI",3841


In [14]:
# outports_dict = dict(zip(outports['WaterwayCode'], ['duluth_superior', 'two_harbors', 'presque_isle', 'marquette_harbor']))
outports_dict = dict(zip(outports['WaterwayCode'], ['duluth_superior', 'two_harbors', 'marquette_harbor']))


In [ ]:
# for code, port in outports_dict.items():
#     trips_port = wcus_all_out.loc[
#         wcus_all_out['WaterwayCode'] == code,
#         cols_keep
#     ]
#     trip_counts_port = trips_port.groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'CompletedYear']).sum().reset_index()

#     trip_counts_port.drop(columns=['VesselDraftFt']).to_csv(
#             '../data/clean/trip_counts/outbound/{}.csv'.format(port),
#             index=False
#         )


In [28]:
wcus_outport_counts = wcus.loc[
    wcus['WaterwayCode'].isin(outports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] != 'Waterway')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['Up/Down']=='Port')
            & (wcus['TrafficCode'] == 1),
    ['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear', 'Trips']
].groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear']).sum().reset_index()

In [42]:
wcus_outport_counts.to_csv(f'../data/clean/trip_counts/wcus_outport_{min_draft_ft}ftplus_counts.csv', index=False)

### Burns Inbound Trip Counts ("")

In [31]:
wcus_inport_counts = wcus.loc[
    wcus['WaterwayCode'].isin(inports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] != 'Waterway')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['Up/Down']=='Port')
            & (wcus['TrafficCode'] == 1),
    ['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear', 'Trips']
].groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear']).sum().reset_index()

In [41]:
wcus_inport_counts.to_csv(f'../data/clean/trip_counts/wcus_inport_{min_draft_ft}ftplus_counts.csv', index=False)

In [ ]:
# wcus_all_in = wcus.loc[
#         wcus['WaterwayCode'].isin(inports['WaterwayCode']) 
#             & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
#             & (wcus['In/Out/Thru'] == 'Inbound Receiving')
#             & (wcus['VesselDraftFt'] >= min_draft_ft)
#             & (wcus['TrafficCode'] == 1) # Domestic (Trips & Drafts)
#             ,
#         :
#     ]
# # wcus_all_in.head()

In [18]:
inports.merge(wcus, how='left')['WaterwayName'].unique()

array(['Indiana Harbor, IN', 'Gary, IN', 'Burns Waterway Harbor, IN',
       'Toledo-Lucas County Port, OH', 'Cleveland-Cuyahoga Port, OH',
       'Rouge River, MI', 'Conneaut Harbor, OH',
       'Ashtabula Port Authority, OH',
       'Calumet Harbor and River, IL and IN', 'Chicago Harbor, IL'],
      dtype=object)

In [ ]:
# trips_burns = wcus_all_in.loc[
#     wcus_all_in['WaterwayName'] == 'Burns Waterway Harbor, IN',
#     cols_keep
# ]
# trip_counts_burns = trips_burns.groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'CompletedYear']).sum().reset_index()

# trip_counts_burns.drop(columns=['VesselDraftFt']).to_csv('../data/clean/trip_counts/inbound/burns.csv')

### Soo Locks through-trips

In [33]:
wcus['In/Out/Thru'].unique()

array(['Waterway', 'Inbound Receiving', 'Outbound Shipping'], dtype=object)

In [36]:
# This is the Sault Ste. Marie river reach, which is not intended for tracking thru-traffic as a waterway
wcus.loc[wcus['WaterwayName'].str.startswith('Sault') , :].head(3)


,RegionName,WaterwayCode,WaterwayName,TrafficCode,TrafficName,In/Out/Thru,Up/Down,VesselType,VesselTypeName,VesselDraftFt,Trips,CompletedYear
188771,GREAT LAKES,3817,"Sault Ste Marie, MI",1,Domestic (Trips & Drafts),Inbound Receiving,Port,4.0,Dry Cargo Barge,1,1,2014
188772,GREAT LAKES,3817,"Sault Ste Marie, MI",1,Domestic (Trips & Drafts),Inbound Receiving,Port,4.0,Dry Cargo Barge,1,1,2016
188773,GREAT LAKES,3817,"Sault Ste Marie, MI",1,Domestic (Trips & Drafts),Inbound Receiving,Port,4.0,Dry Cargo Barge,1,1,2020


In [37]:
wcus_soo = wcus.loc[
        wcus['WaterwayCode'].isin([3811]) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] == 'Waterway')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['TrafficCode'] == 1), 
        :
    ]
wcus_soo.head(3)

,RegionName,WaterwayCode,WaterwayName,TrafficCode,TrafficName,In/Out/Thru,Up/Down,VesselType,VesselTypeName,VesselDraftFt,Trips,CompletedYear
193512,GREAT LAKES,3811,"St. Marys Falls Canal, MI and Sault Ste. Marie...",1,Domestic (Trips & Drafts),Waterway,Downbound/West/South,1.0,Self-Propelled dry,25,46,2014
193513,GREAT LAKES,3811,"St. Marys Falls Canal, MI and Sault Ste. Marie...",1,Domestic (Trips & Drafts),Waterway,Downbound/West/South,1.0,Self-Propelled dry,25,30,2015
193514,GREAT LAKES,3811,"St. Marys Falls Canal, MI and Sault Ste. Marie...",1,Domestic (Trips & Drafts),Waterway,Downbound/West/South,1.0,Self-Propelled dry,25,28,2016


In [38]:
wcus_soo_counts = wcus_soo.loc[
    :,
    ['WaterwayCode', 'WaterwayName', 'Up/Down', 'CompletedYear', 'Trips']
].groupby(['WaterwayCode', 'WaterwayName', 'Up/Down', 'CompletedYear']).sum().reset_index()

In [53]:
wcus_soo_counts.loc[
    wcus_soo_counts['CompletedYear']!=2023,
    :
].groupby(by=['WaterwayCode', 'WaterwayName', 'Up/Down']).mean()

CompletedYear  \
WaterwayCode WaterwayName                                       Up/Down                               
3811         St. Marys Falls Canal, MI and Sault Ste. Marie,... Downbound/West/South         2018.0   
                                                                Upbound/East/North           2018.0   

                                                                                           Trips  
WaterwayCode WaterwayName                                       Up/Down                           
3811         St. Marys Falls Canal, MI and Sault Ste. Marie,... Downbound/West/South  680.222222  
                                                                Upbound/East/North    100.333333

In [54]:
(381-680.222222)/680.222222

-0.43988892500486404

In [55]:
(77-100.333333)/100.333333

-0.23255813698524294

In [40]:
wcus_soo_counts.to_csv(f'../data/clean/trip_counts/wcus_soo_{min_draft_ft}ftplus_counts.csv', index=False)

## WCUS Cargo cleaning steps

1. Filter to only WaterwayCodes in `inports` and `outports`
2. `CommodityCode` 4410 (iron ore)
3. Only Lakewise vessel movements (no intraport movements, for example)

## WCUS Cargo cleaning

In [18]:
path = "../data/raw/Cargo_AllRegions_10yr_2013-2022.xlsx"

cargo = pd.read_excel(path, sheet_name="Cargo_AllRegions_10yr_2013-2022")

In [19]:
cols_to_drop_cargo = ['TrafficCode', 'Allo1Code', 'Allo2Code', 'Up/Down', 'TonMiles']

### Outbound

In [48]:
cargo.loc[cargo['WaterwayName'].str.startswith('Taconite') , :].head(3)


,WaterwayCode,WaterwayName,TrafficCode,TrafficName,CommodityCode,CommodityName,Allo1Code,In/Out/Thru,Allo2Code,Up/Down,ShortTons,TonMiles,CompletedYear
270085,3929,"Taconite, MN",21,Canadian Imports,4410,Iron Ore,1,Inbound Receiving,0,Port,32068,0,2018
270086,3929,"Taconite, MN",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,33069,0,2017
270087,3929,"Taconite, MN",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,31400,0,2018


In [49]:
cargo.loc[
            (cargo['WaterwayCode']==3929)
            & (cargo['CommodityCode']==4410)
            & (cargo['TrafficName']=='Lakewise')
            & (cargo['In/Out/Thru']=='Outbound Shipping'), 
            :
        ].drop(columns = cols_to_drop_cargo)

,WaterwayCode,WaterwayName,TrafficName,CommodityCode,CommodityName,In/Out/Thru,ShortTons,CompletedYear
270094,3929,"Taconite, MN",Lakewise,4410,Iron Ore,Outbound Shipping,20397,2014
270095,3929,"Taconite, MN",Lakewise,4410,Iron Ore,Outbound Shipping,23440,2017
270096,3929,"Taconite, MN",Lakewise,4410,Iron Ore,Outbound Shipping,18154,2019
270097,3929,"Taconite, MN",Lakewise,4410,Iron Ore,Outbound Shipping,321312,2021
270098,3929,"Taconite, MN",Lakewise,4410,Iron Ore,Outbound Shipping,493029,2022


In [255]:
for code, port in outports_dict.items():
    
    cargoes_port = cargo.loc[
            (cargo['WaterwayCode']==code)
            & (cargo['CommodityCode']==4410)
            & (cargo['TrafficName']=='Lakewise')
            & (cargo['In/Out/Thru']=='Outbound Shipping'), 
            :
        ].drop(columns = cols_to_drop_cargo)


    cargoes_port.to_csv(
            '../data/clean/cargoes/outbound/{}.csv'.format(port),
            index=False
        )


In [20]:
cargoes_outport = cargo.loc[
            (cargo['WaterwayCode'].isin(outports['WaterwayCode']))
            & (cargo['CommodityCode']==4410)
            & (cargo['TrafficName']=='Lakewise')
            & (cargo['In/Out/Thru']=='Outbound Shipping'), 
            :
        ].drop(columns = cols_to_drop_cargo)
# cargoes_outport.head()

In [21]:
cargoes_outport.to_csv('../data/clean/cargoes/wcus_outport_cargoes.csv')

In [232]:
cargo.loc[(cargo['WaterwayCode']==3619) & (cargo['CommodityName']=='Iron Ore'), :]

,WaterwayCode,WaterwayName,TrafficCode,TrafficName,CommodityCode,CommodityName,Allo1Code,In/Out/Thru,Allo2Code,Up/Down,ShortTons,TonMiles,CompletedYear
257102,3619,"Presque Isle Township, MI",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,21076,0,2015
257103,3619,"Presque Isle Township, MI",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,17000,0,2016
257104,3619,"Presque Isle Township, MI",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,22,0,2017


This explains that Presque Isle only exports to Canada! No domestic shipments of iron ore from here.

### Inbound

In [9]:
cargo.columns

Index(['WaterwayCode', 'WaterwayName', 'TrafficCode', 'TrafficName',
       'CommodityCode', 'CommodityName', 'Allo1Code', 'In/Out/Thru',
       'Allo2Code', 'Up/Down', 'ShortTons', 'TonMiles', 'CompletedYear'],
      dtype='object')

In [56]:
# Which inport has historically received the most ore on average?
cargo.loc[
    cargo['WaterwayCode'].isin(inports['WaterwayCode'])
        & (cargo['CommodityCode']==4410)
        & (cargo['TrafficName']=='Lakewise')
        & (cargo['In/Out/Thru']=='Inbound Receiving')
    ,
    ['WaterwayCode', 'WaterwayName', 'ShortTons']
].groupby(by=['WaterwayCode', 'WaterwayName']).mean().sort_values(by='ShortTons', ascending=False)

,,ShortTons
WaterwayCode,WaterwayName,
3738,"Indiana Harbor, IN",7958927.8
3736,"Gary, IN",7595848.3
3739,"Burns Waterway Harbor, IN",5497330.4
3217,"Cleveland-Cuyahoga Port, OH",3921239.0
3220,"Conneaut Harbor, OH",3682045.9
3204,"Toledo-Lucas County Port, OH",3314558.1
3315,"Rouge River, MI",3192452.0
3219,"Ashtabula Port Authority, OH",959381.0


In [83]:
wcus['In/Out/Thru'].unique()

array(['Waterway', 'Inbound Receiving', 'Outbound Shipping'], dtype=object)

In [88]:
cargoes_inport = cargo.loc[
            (cargo['WaterwayCode'].isin(inports['WaterwayCode']))
            & (cargo['CommodityCode']==4410)
            & (cargo['TrafficName']=='Lakewise')
            & (cargo['In/Out/Thru']=='Inbound Receiving'), 
            :
        ].drop(columns = cols_to_drop_cargo)
# cargoes_inport.head()

In [89]:
cargoes_inport.to_csv('../data/clean/cargoes/wcus_inport_cargoes.csv')

### Soo Locks

In [103]:
cargoes_soo = cargo.loc[
            cargo['WaterwayCode'].isin([3811])
                & (cargo['CommodityCode']==4410)
                # & (cargo['In/Out/Thru']=='Inbound Receiving')
                & (cargo['TrafficName']=='Lakewise'),
            :
        ]#.drop(columns = cols_to_drop_cargo)
# cargoes_soo.head()

In [106]:
cargoes_soo.head(3)

,WaterwayCode,WaterwayName,TrafficCode,TrafficName,CommodityCode,CommodityName,Allo1Code,In/Out/Thru,Allo2Code,Up/Down,ShortTons,TonMiles,CompletedYear
268204,3811,"St. Marys Falls Canal, MI and Sault Ste. Marie...",40,Lakewise,4410,Iron Ore,4,Thru,1,Upbound/East/North,17000,17000,2013
268205,3811,"St. Marys Falls Canal, MI and Sault Ste. Marie...",40,Lakewise,4410,Iron Ore,4,Thru,1,Upbound/East/North,133678,133678,2014
268206,3811,"St. Marys Falls Canal, MI and Sault Ste. Marie...",40,Lakewise,4410,Iron Ore,4,Thru,1,Upbound/East/North,52007,52007,2017


In [107]:
cargoes_soo.to_csv('../data/clean/cargoes/wcus_soo_cargoes.csv')

## Crosswalk between WaterwayCode and USACE Port Statistical Areas

In [258]:
# test crosswalk
cw = pd.read_csv('../data/crosswalks/port_id_to_waterwaycode_outbound.csv')

In [259]:
cw

,WaterwayCode,OBJECTID,shortname
0,3924,3,duluth_superior
1,3841,270,marquette_harbor
2,3926,35,two_harbors
3,3845,270,presque_isle
